 <a target="_blank" href="https://colab.research.google.com/github/mbari-org/stm/blob/main/stm/notebooks/train_topic_model_pcen.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
 Author: Danelle Cline dcline@mbari.org

## Train a topic model with PCEN features
---
This notebook extracts PCEN spectrogram features and trains a topic model to uncover recurring sound phrases without using Perch2 embeddings. Document length is derived from the STFT hop, sample rate, and words-per-document setting.

Author: Danelle Cline dcline@mbari.org

### Install the ROST topic model
Install build dependencies, clone rost-cli, and compile the binaries. This follows the same steps as the ROST Docker image. The first run takes several minutes; later runs skip the build if `topics.refine.t` is already present.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["DEBIAN_FRONTEND"] = "noninteractive"
os.environ["ROSTPATH"] = "/share/rost"
os.environ["PATH"] = "/app/rost-cli/bin:" + os.environ.get("PATH", "")

rost_root = Path("/app/rost-cli")
rost_bin = rost_root / "bin" / "topics.refine.t"
Path("/share/rost").mkdir(parents=True, exist_ok=True)
rost_root.parent.mkdir(parents=True, exist_ok=True)

def sh(command: str, cwd: str | None = None) -> None:
    print(command, flush=True)
    subprocess.check_call(command, shell=True, cwd=cwd)

apt = "apt-get" if os.geteuid() == 0 else "sudo apt-get"

if rost_bin.exists():
    print(f"ROST already installed: {rost_bin}")
else:
    sh(
        f"{apt} update && {apt} install -y "
        "git cmake build-essential libboost-all-dev libflann-dev "
        "libfftw3-dev libopencv-dev libsndfile1-dev "
        "libgstreamer-plugins-base1.0-dev libgstreamer1.0-0 python3-pip"
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pysrt"])
    if not (rost_root / ".git").exists():
        sh(f"git clone https://gitlab.com/warplab/rost-cli.git {rost_root}")
    sh("mkdir -p build && cd build && cmake .. && make -j4 && make install", cwd=str(rost_root))
    print(f"ROST installed: {rost_bin}")


### Load configuration
Create the output directory, then load and verify `config.yaml`. The config supplies PCEN settings, topic-model hyperparameters, and paths used in later cells.

In [ ]:
from pathlib import Path

output_path = Path("output_pcen")
dataset_path = Path("dataset")
output_path.mkdir(parents=True, exist_ok=True)

from stm.config import Config
try:
    config = Config.load(
        Path("../config.yaml"),
        wav_path=dataset_path,
        output_path=output_path,
    )
    config.verify()
except Exception as e:
    print(e)

### Extract PCEN features
Measure the dataset duration and extract PCEN spectrogram features. These become the words of the topic model.

In [ ]:
from stm.topicmodel import TopicModelRunner
from stm.features import PcenExtractor
from stm.embed import total_audio_seconds

duration = total_audio_seconds(dataset_path)
n = total_audio_seconds(dataset_path)
n_windows = int((duration - config.perch_audio_seconds) // config.perch_hop_seconds) + 1


print(f"Processing {duration} total audio seconds")
pcen_block = PcenExtractor.from_config(config).extract(dataset_path)

### Train the topic model
Compute document duration from the STFT hop and `word_per_doc`, then fit a topic model on the PCEN feature block.

In [ ]:
document_seconds = config.window_size * (1 - config.overlap) / 32e3 * config.word_per_doc
print(f"Topic model documents {document_seconds} audio seconds")
runner = TopicModelRunner(
    timeout=600,
    alpha=config.alpha,
    beta=config.beta,
    gamma=config.gamma,
    num_topics=None,
    document_seconds=document_seconds,
    use_docker=False,
    rost_path=Path("/app/rost-cli/bin"),
)
result = runner.run_from_block(
    [pcen_block],
    config.doc_path,
    config.model_path,
    target=pcen_block.grid,
)

### Report model diagnostics
Print average perplexity and the path to the maximum-likelihood topic-over-time file.

In [ ]:
print(result.avg_perplexity, result.maxlikelihood_with_time_path)


### Plot topics on spectrograms
Overlay inferred topics on spectrogram chunks of the source audio for visual inspection.

In [ ]:
from stm.topicmodel.plotter import Plotter

plotter = Plotter(model_dir=config.model_path, config=config)
plotter.plot(dataset_path, n_chunks=10, chunk_size=60, freq_range=(0, 4000), window_size=1024)